In [1]:
# 05_revision_experiments.ipynb
# (1) interior delegation under convex error exposure (Prop 4.2)  -> fig8
# (2) joint allocation-staffing-friction program (Section 4.4)    -> fig7, joint_opt.json
# (3) bottleneck-migration numbers at the joint optimum
# (4) seasonal stress test with the fixed scenario mixture         -> fig9

## Shared model

In [2]:
# ---- shared model block (identical in 05 and 06) ----
import itertools, json
import numpy as np, pandas as pd
df = pd.read_csv("../data/tasks.csv")
TIDS = df["task_id"].tolist()
tau = df["pre_ai_hours"].to_numpy(float)          # human task time; AI draft replaces it: g_i = tau_i
v = df["verification_hours"].to_numpy(float)       # time to verify a full AI draft
sev = df["error_severity"].to_numpy(float)         # 1-5 rating
e = sev / 5.0                                      # error cost of a fully delegated task (hour-equivalents)
cap = np.where(df["accountability_constraint"] == 1, 0.5, 1.0)   # accountability caps
LAM, K = 0.30, 0.6                                 # illustrative arrival rate (cases/h), friction sensitivity
W_BASE = dict(c_H=0.4, w_h=0.25, w_e=5.0)          # w_e * e_i = severity
LEVELS = (0.0, 0.25, 0.5, 0.75, 1.0)
X = np.array(list(itertools.product(*[[l for l in LEVELS if l <= cap[i]] for i in range(len(TIDS))])))
H = ((1 - X) * tau).sum(1)                         # retained human time
V = (X * v).sum(1)                                 # verification of delegated output
P = (e * X**2).sum(1)                              # convex error exposure, phi(x) = x^2
FS = np.round(np.arange(0, 12.0001, 0.05), 2)

def best_at_f(f, c_H=0.4, w_h=0.25, w_e=5.0):
    """Min over (x, c) at fixed friction f. Friction acts on v_i(f) = v_i(1 + k f) only;
    c is the smallest integer with c > lam E[S]; escaped-error cost falls as 1/(1+f)."""
    ES = H + V * (1 + K * f)
    c = np.floor(LAM * ES) + 1
    Z = c_H * c + w_h * ES + w_e * P / (1 + f)
    j = int(Z.argmin())
    return float(Z[j]), int(c[j]), X[j], float(ES[j])

def solve(**w):
    env = [best_at_f(f, **w) for f in FS]
    j = int(np.argmin([r[0] for r in env]))
    return float(FS[j]), env[j], env


## (1) Task-level optimal delegation: linear vs convex exposure

In [3]:
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({"savefig.dpi":600,"font.size":11,"axes.edgecolor":"0.2","axes.linewidth":0.8,"grid.color":"0.85"})
def save(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(f"../results/figures/{name}.{ext}", dpi=600, bbox_inches="tight")
    plt.close(fig)
g = tau.copy()
x_lin = np.minimum(np.where(g - v - e > 0, 1.0, 0.0), cap)
x_cvx = np.minimum(np.clip((g - v) / (2*e), 0, 1), cap)
tab = pd.DataFrame(dict(task=TIDS, g=g, v=v, e=e, cap=cap, x_linear=x_lin, x_convex=x_cvx)); print(tab.to_string(index=False))
order = [2, 5, 3, 4, 0, 1, 6]; xp = np.arange(7)
fig, ax = plt.subplots(figsize=(6, 4.2))
ax.bar(xp-0.2, x_lin[order], width=0.4, color="0.75", edgecolor="black", lw=0.6, label="linear $\\varphi(x)=x$ (bang-bang)")
ax.bar(xp+0.2, x_cvx[order], width=0.4, color="0.3", edgecolor="black", lw=0.6, label="convex $\\varphi(x)=x^2$ (interior)")
for k_, i in enumerate(order):
    if 0 < x_cvx[i] < cap[i]: ax.annotate(f"{x_cvx[i]:.3f}", (k_+0.2, x_cvx[i]), textcoords="offset points", xytext=(0, 3), ha="center", fontsize=8)
ax.set_xticks(xp); ax.set_xticklabels([TIDS[i] for i in order]); ax.set_ylim(0, 1.3)
ax.set_ylabel("Optimal delegation $x_i^*$"); ax.set_xlabel("Task")
ax.legend(frameon=True, edgecolor="0.5", fontsize=9, loc="upper center", ncol=2)
save(fig, "fig8_nonlinear_delegation")

task   g   v   e  cap  x_linear  x_convex
  T1 2.5 0.1 0.4  1.0       1.0     1.000
  T2 3.0 0.2 0.6  1.0       1.0     1.000
  T3 1.5 2.5 1.0  0.5       0.0     0.000
  T4 1.0 0.3 0.4  1.0       1.0     0.875
  T5 2.0 0.2 0.8  1.0       1.0     1.000
  T6 0.5 1.5 1.0  0.5       0.0     0.000
  T7 1.5 0.1 0.6  0.5       0.5     0.500


## (2) Joint program: envelope over friction

In [4]:
f_star, (z_star, c_star, x_star, es_star), env = solve(**W_BASE)
print(f"f* = {f_star:.2f}  c* = {c_star}  Z* = {z_star:.4f}  E[S](f*) = {es_star:.3f} h")
print("x* =", dict(zip(TIDS, x_star)))
# friction level at which the optimal policy would need one more reviewer
thr = next(float(f) for f, r in zip(FS, env) if f > f_star and r[1] > c_star)
print("next reviewer needed from f =", thr)
fig, ax = plt.subplots(figsize=(6, 4.2))
ax.plot(FS, [r[0] for r in env], color="0.15", lw=1.7)
ax.axvline(f_star, color="0.5", ls="--", lw=1.2)
ax.plot([f_star], [z_star], "o", color="0.1", markersize=8)
ax.annotate(f"$f^*={f_star:.2f}$", (f_star, z_star), textcoords="offset points", xytext=(-60, -4), fontsize=10)
ax2 = ax.twinx(); ax2.step(FS, [r[1] for r in env], where="post", color="0.6", lw=1.0, ls=":")
ax2.set_ylabel("Reviewers $c^*$ (dotted)"); ax2.set_yticks([1, 2, 3, 4]); ax2.grid(False)
ax.set_xlabel("Forced-friction intensity $f$"); ax.set_ylabel("Minimised total cost $Z(x^*,c^*,f)$")
save(fig, "fig7_joint_optimum")
json.dump(dict(f=f_star, c=c_star, cost=round(z_star, 4), ES_at_fstar=round(es_star, 3),
               x=dict(zip(TIDS, map(float, x_star))), next_reviewer_from_f=thr,
               params=dict(lam=LAM, k=K, **W_BASE)), open("../results/tables/joint_opt.json", "w"), indent=2)

f* = 5.40  c* = 2  Z* = 3.5192  E[S](f*) = 6.658 h
x* = {'T1': np.float64(1.0), 'T2': np.float64(1.0), 'T3': np.float64(0.0), 'T4': np.float64(0.0), 'T5': np.float64(0.5), 'T6': np.float64(0.0), 'T7': np.float64(0.5)}
next reviewer needed from f = 7.2


## (3) Bottleneck migration at the optimum

In [5]:
pre = tau.sum()
B = float(((1 - x_star)*tau + x_star*v).sum())
add = float(sum(v[i] - tau[i] for i in (2, 5)))
print(f"pre-AI human load {pre:.1f} h | review load at x*: {B:.2f} h before friction, {es_star:.2f} h at f* | delegating T3,T6 adds {add:.1f} h (f=0)")

pre-AI human load 12.0 h | review load at x*: 5.20 h before friction, 6.66 h at f* | delegating T3,T6 adds 2.0 h (f=0)


## (4) Seasonal stress test

In [6]:
qp = json.load(open("../data/queue_params.json")); p = qp["client_mix"]["prime_share"]
sf = qp["service_time_type_means"]["s_fast_prime_hours"]; ss = qp["service_time_type_means"]["s_slow_scarce_hours"]
ESq = p*sf + (1-p)*ss; c = 12; base = 0.5*c/ESq      # base load rho = 0.5
def sim_ns(base, pm, frac, seed=0, horizon=8000):
    rng = np.random.default_rng(seed); free = [0.0]*c; t = 0.0; W = []; n = 0
    while n < horizon:
        cl = base*pm if (t % 100)/100 < frac else base
        t += rng.exponential(1/cl); svc = sf if rng.random() < p else ss
        j = int(np.argmin(free)); st = max(t, free[j]); free[j] = st + svc
        if n > 500: W.append(st - t)
        n += 1
    return np.array(W)
mult = [1, 2, 3]; means = []; p95 = []
for pm in mult:
    allW = np.concatenate([sim_ns(base, pm, 0.25, seed=s) for s in range(20)])
    means.append(float(allW.mean())); p95.append(float(np.percentile(allW, 95)))
print("mean", np.round(means, 3), "p95", np.round(p95, 2))
fig, ax = plt.subplots(figsize=(6, 4.2)); xp = np.arange(3)
ax.bar(xp-0.2, means, width=0.4, color="0.6", edgecolor="black", lw=0.6, label="mean $W_q$")
ax.bar(xp+0.2, p95, width=0.4, color="0.25", edgecolor="black", lw=0.6, label="95th percentile $W_q$")
ax.set_xticks(xp); ax.set_xticklabels([f"×{m}" for m in mult])
ax.set_xlabel("Seasonal peak multiplier"); ax.set_ylabel("Waiting time $W_q$ (h)"); ax.legend(frameon=True, edgecolor="0.5")
save(fig, "fig9_seasonality")
json.dump(dict(migration=dict(pre=float(pre), review_before_friction=B, review_at_fstar=es_star, add_loss_tasks=add),
               season=dict(mult=mult, mean=means, p95=p95)), open("../results/tables/revision_results.json", "w"), indent=2)

mean [0.013 0.538 3.982] p95 [ 0.    3.46 13.57]
